# Level by level: which levels of the tree does each RDM recover?

**Question.** For each RDM, at which levels of the hierarchy (domain, mid-level, basic level) are images of different categories further apart than images of the same category?

**Why it matters.** Pre-registered H1b says the effect is strongest at the superordinate levels and weakest at the basic level (Mur et al., 2013). Notebook `2_rank_agreement` asks how much rank structure an RDM shares with the tree overall; this notebook asks where in the tree that agreement lives. The old notebook found Cohen's d *rising* with depth, the opposite of H1b. Here we check whether that is partly an artifact of how "between" is defined (4.2), and put bootstrap CIs on everything (4.3).

**RDM set.** `sem_km` (the tree itself), `sem_wn`, `clip_pre`, `clip_post`, `sens_pre`, `sens_post`, `behav_joint` (raw, ~13% of pairs unobserved and dropped pairwise), and its two SMACOF embeddings `behav_joint_mds_low` (stress-elbow dimensionality) and `behav_joint_mds_high` (95% of held-out gain), once notebook 1 has built them (the first code cell falls back to the raw set with a printed note otherwise).

**Run from this directory** with the RDMs built.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s2_semantic_hierarchy/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from analysis.s2_semantic_hierarchy import common as c
from analysis.s2_semantic_hierarchy import level_separation as ls

N_BOOT = 1000
CACHE = c.S2_RESULTS_DIR

try:
    RDMS = c.load_rdm_set(c.FULL_SET)
except FileNotFoundError as err:
    print(f"NOTE: {err}\nFalling back to the raw RDM set (no behav_joint_mds_low or behav_joint_mds_high).")
    RDMS = c.load_rdm_set(c.RAW_SET)
print(list(RDMS))
for level, name in c.LEVEL_NAMES.items():
    n_cat = len({p[:level] for p in c.tree_parts()})
    print(f"level {level} ({name}): {n_cat} categories")

['sem_km', 'sem_wn', 'clip_pre', 'clip_post', 'sens_pre', 'sens_post', 'behav_joint', 'behav_joint_mds_low', 'behav_joint_mds_high']
level 1 (domain (animate / inanimate)): 2 categories
level 2 (mid-level category): 4 categories
level 3 (basic level): 16 categories


## 4.1 Global contrast: does the RDM separate categories at each level?

At level L, "within" = pairs of images in the same level-L category and "between" = every other pair. Cohen's d is the gap between the two means in units of their pooled SD (positive = the RDM recovers that level). Level 1 is animate vs inanimate (2 categories), level 2 has 4 mid-level categories, level 3 has 16 basic-level categories.

In [2]:
global_tab = pd.concat({lv: ls.level_separation(lv, RDMS) for lv in ls.LEVELS}, names=["level"])
print("Cohen's d, global contrast (rows: RDM, columns: level)")
print(global_tab["cohens_d"].unstack("level").round(2).to_string())
print("\npairs per group:")
print(global_tab.loc[:, ["n_within", "n_between"]].groupby("level").first().to_string())

Cohen's d, global contrast (rows: RDM, columns: level)
level                    1     2     3
rdm                                   
sem_km                2.80  3.54  2.85
sem_wn                0.44  0.85  1.28
clip_pre              0.80  0.96  1.80
clip_post             0.72  0.89  1.67
sens_pre              0.17  0.19  0.53
sens_post             0.08  0.20  0.64
behav_joint           0.61  1.02  1.58
behav_joint_mds_low   1.16  2.24  2.95
behav_joint_mds_high  1.24  2.26  2.94

pairs per group:
       n_within  n_between
level                     
1        135204     127246
2         70508     191942
3         21236     241214


**Reading.** Every stimulus-derived RDM and the behavioral RDM show d *increasing* with depth: CLIP pre 0.80, 0.96, 1.80; behavior 0.61, 1.02, 1.58; WordNet 0.44, 0.85, 1.28; pixels (pre) 0.17, 0.19, 0.53; the two MDS embeddings of behavior 1.16, 2.24, 2.95 (low) and 1.24, 2.26, 2.94 (high). That is the opposite of H1b. Only `sem_km` is non-monotonic (2.80, 3.54, 2.85), and it is the reference, so its large d is by construction. The next section asks whether the increase is real or comes from how "between" is built.

## 4.2 Nested contrast: compare siblings only

**Rationale.** In the global contrast the "between" set at the basic level also contains animate vs inanimate pairs (and cross-mid-level pairs), which almost any RDM separates easily. That inflates d at the deep levels, because the within-category group is small and tight while the between group is dominated by far-apart pairs. The nested contrast removes this: at level L, "between" = pairs that share their level L-1 ancestor but differ at level L (siblings), "within" = pairs in the same level-L category. For L = 1 the parent is the root, so nested equals global. The d definition is unchanged.

In [3]:
nested_tab = pd.concat({lv: ls.level_separation(lv, RDMS, nested=True) for lv in ls.LEVELS}, names=["level"])
print("Cohen's d, nested contrast (rows: RDM, columns: level)")
print(nested_tab["cohens_d"].unstack("level").round(2).to_string())
print("\npairs per group:")
print(nested_tab.loc[:, ["n_within", "n_between"]].groupby("level").first().to_string())

Cohen's d, nested contrast (rows: RDM, columns: level)
level                    1     2     3
rdm                                   
sem_km                2.80  2.57  1.36
sem_wn                0.44  0.90  0.79
clip_pre              0.80  0.73  1.36
clip_post             0.72  0.70  1.25
sens_pre              0.17  0.12  0.49
sens_post             0.08  0.21  0.62
behav_joint           0.61  0.94  0.93
behav_joint_mds_low   1.16  2.06  1.53
behav_joint_mds_high  1.24  2.07  1.66

pairs per group:
       n_within  n_between
level                     
1        135204     127246
2         70508      64696
3         21236      49272


**Reading.** Restricting to siblings shrinks the between group (191,942 to 64,696 pairs at level 2, 241,214 to 49,272 at level 3) and lowers d at the deep levels: at the basic level CLIP pre 1.80 to 1.36, behavior 1.58 to 0.93, WordNet 1.28 to 0.79, `sem_km` 2.85 to 1.36. So part of the old rise with depth was the artifact. It does not disappear: CLIP (0.80, 0.73, 1.36) and pixels (pre: 0.17, 0.12, 0.49) still have their largest d at the basic level, while raw behavior (0.61, 0.94, 0.93), the MDS embeddings (low 1.16, 2.06, 1.53; high 1.24, 2.07, 1.66) and WordNet (0.44, 0.90, 0.79) peak at the mid-level. The embeddings raise d at every level relative to raw behavior, but keep its mid-level peak. Whether these differences are reliable is the job of 4.3.

## 4.3 Bootstrap CIs and the H1b ordering test

Stimuli are the sampling unit, so every d (global and nested) gets a 95% percentile CI from 1000 image-bootstrap draws. The per-pair "same category at level L" indicators are resampled together with the distances (as extra condensed arrays), so group membership stays consistent in each draw. H1b predicts d(L1) > d(L2) and d(L2) > d(L3); we test both on the nested contrast through the bootstrap distribution of each difference (CI, and the fraction of draws in the predicted direction). The global contrast is shown for comparison.

In [4]:
table = ls.separation_table(RDMS, n_boot=N_BOOT, cache_dir=CACHE)
show = table.assign(txt=lambda t: t["d"].map("{:.2f}".format) + " [" + t["ci_lo"].map("{:.2f}".format) + ", " + t["ci_hi"].map("{:.2f}".format) + "]")
for contrast in ls.CONTRASTS:
    print(f"d [95% CI], {contrast} contrast")
    print(show[show["contrast"] == contrast].pivot(index="rdm", columns="level", values="txt").loc[list(RDMS)].to_string())
    print()
ls.fig_levels(table).show()

d [95% CI], nested contrast
level                                 1                  2                  3
rdm                                                                          
sem_km                2.80 [2.65, 2.98]  2.57 [2.44, 2.74]  1.36 [1.26, 1.49]
sem_wn                0.44 [0.37, 0.52]  0.90 [0.80, 1.01]  0.79 [0.66, 0.93]
clip_pre              0.80 [0.72, 0.89]  0.73 [0.63, 0.84]  1.36 [1.25, 1.47]
clip_post             0.72 [0.64, 0.80]  0.70 [0.60, 0.80]  1.25 [1.16, 1.37]
sens_pre              0.17 [0.11, 0.24]  0.12 [0.06, 0.20]  0.49 [0.39, 0.61]
sens_post             0.08 [0.03, 0.15]  0.21 [0.14, 0.31]  0.62 [0.49, 0.74]
behav_joint           0.61 [0.58, 0.65]  0.94 [0.87, 1.02]  0.93 [0.84, 1.02]
behav_joint_mds_low   1.16 [1.09, 1.23]  2.06 [1.88, 2.26]  1.53 [1.39, 1.68]
behav_joint_mds_high  1.24 [1.19, 1.30]  2.07 [1.92, 2.22]  1.66 [1.51, 1.82]

d [95% CI], global contrast
level                                 1                  2                  3
rdm    

**Reading the figure and table.** CIs are narrow relative to the differences between levels for most RDMs (nested CLIP pre: 0.80 [0.72, 0.89], 0.73 [0.63, 0.84], 1.36 [1.25, 1.47]). The nested panel is flatter than the global one, but its basic-level point is still the highest for CLIP and pixels. The behavioral RDM's CIs are noticeably narrower than the others'.

In [5]:
order = ls.ordering_table(RDMS, n_boot=N_BOOT, cache_dir=CACHE)
order["d [95% CI]"] = order["diff"].map("{:.2f}".format) + " [" + order["ci_lo"].map("{:.2f}".format) + ", " + order["ci_hi"].map("{:.2f}".format) + "]"
order["frac_predicted"] = order["frac_predicted"].round(3)
for contrast in ls.CONTRASTS:
    print(f"H1b ordering, {contrast} contrast (predicted: both differences > 0)")
    sub = order[order["contrast"] == contrast].pivot(index="rdm", columns="step", values=["d [95% CI]", "frac_predicted"])
    print(sub.loc[list(RDMS)].to_string())
    print()

H1b ordering, nested contrast (predicted: both differences > 0)
                                d [95% CI]                       frac_predicted        
step                               d1 - d2               d2 - d3        d1 - d2 d2 - d3
rdm                                                                                    
sem_km                   0.24 [0.06, 0.40]     1.20 [1.02, 1.38]          0.997     1.0
sem_wn                -0.46 [-0.57, -0.35]    0.11 [-0.06, 0.28]            0.0   0.891
clip_pre                0.07 [-0.05, 0.20]  -0.63 [-0.78, -0.47]           0.86     0.0
clip_post               0.01 [-0.10, 0.13]  -0.55 [-0.71, -0.40]          0.585     0.0
sens_pre                0.05 [-0.04, 0.14]  -0.37 [-0.50, -0.23]          0.843     0.0
sens_post             -0.13 [-0.22, -0.05]  -0.41 [-0.54, -0.25]            0.0     0.0
behav_joint           -0.33 [-0.40, -0.26]    0.01 [-0.10, 0.12]            0.0   0.576
behav_joint_mds_low   -0.90 [-1.11, -0.72]     0.53 [0.2

**H1b ordering, nested contrast.**

* `sem_km` (reference): both steps in the predicted direction (d1 - d2 = 0.24 [0.06, 0.40], 99.7% of draws; d2 - d3 = 1.20 [1.02, 1.38], 100%). The tree obeys H1b once siblings are compared, but not globally (d1 - d2 = -0.73).
* `sem_wn`: step 1 reversed (-0.46 [-0.57, -0.35]); step 2 not distinguishable from 0 (0.11 [-0.06, 0.28], 89% of draws).
* `clip_pre`, `clip_post`: step 1 not distinguishable from 0 (0.07 [-0.05, 0.20] and 0.01 [-0.10, 0.13]); step 2 reversed (-0.63 [-0.78, -0.47] and -0.55 [-0.71, -0.40]).
* `sens_pre`: step 1 not distinguishable from 0 (0.05 [-0.04, 0.14]); step 2 reversed (-0.37 [-0.50, -0.23]). `sens_post`: both reversed (-0.13 [-0.22, -0.05] and -0.41 [-0.54, -0.25]).
* `behav_joint`: step 1 reversed (-0.33 [-0.40, -0.26]); step 2 flat (0.01 [-0.10, 0.12]).
* `behav_joint_mds_low`, `behav_joint_mds_high`: step 1 reversed (-0.90 [-1.11, -0.72] and -0.83 [-1.00, -0.67]); step 2 in the predicted direction (0.53 [0.28, 0.79] and 0.42 [0.18, 0.65]), i.e. mid-level above basic level, but the domain split is the weakest level.

No RDM other than the tree itself satisfies both predicted steps. Under the global contrast even the tree does not (non-monotonic), and every other RDM reverses both steps (e.g. MDS low -1.08 [-1.23, -0.92] and -0.72 [-0.93, -0.53]).

## 4.4 Which basic-level categories does each RDM separate worst? (descriptive)

Per basic-level category: nested d between the category (within = its own pairs) and its siblings (pairs of one of its images with an image of another basic-level category under the same mid-level parent). No inference; this only localizes the pooled numbers above. `sem_km` d varies across categories (4.1 to 7.1) because the ragged tree gives categories different KM distances.

In [6]:
by_category = ls.category_separation(RDMS)
ls.fig_categories(by_category).show()
print(by_category.round(2).to_string())
print("\nweakest category per RDM:")
print(by_category.idxmin().to_string())

                              sem_km  sem_wn  clip_pre  clip_post  sens_pre  sens_post  behav_joint  behav_joint_mds_low  behav_joint_mds_high
category                                                                                                                                      
animate/animal/body             4.12    0.47      0.40       0.37      0.74       1.11         0.19                 0.41                  0.18
animate/animal/face             4.58    0.13      0.80       0.77      0.30       0.13         0.55                 1.04                  1.02
animate/human/body              7.14   -0.09      0.89       0.78      1.43       1.08         0.42                 0.82                  0.69
animate/human/face              6.96    1.41      1.87       1.64      1.41       1.75         1.36                 3.73                  4.99
inanimate/handmade/ball         5.57    0.76      2.02       2.16      0.16       0.22         1.73                 2.88                  3.74

**Reading.** The weakest category differs by RDM, but a catch-all recurs: `inanimate/handmade/other` is weakest for CLIP pre and post (0.03, 0.12) and for behavior (0.08), plausible for a heterogeneous "other" bucket. Pixel RDMs are weakest where members share no low-level look (`handmade/electric` -0.10 for pre, `natural/food` -0.27 for post), and WordNet is weakest for `animate/human/body` (-0.09) while strongest for `handmade/food` (4.87). Human faces are well separated by all non-reference RDMs (1.36 to 1.87), and behavior is strongest for `handmade/ball` (1.73). Small categories rest on few pairs, so treat single cells with caution.

## 4.5 Takeaways

1. **Global contrast: d rises with depth for every data-derived RDM**, e.g. CLIP pre 0.80, 0.96, 1.80 and behavior 0.61, 1.02, 1.58 (levels 1 to 3), the opposite of H1b.
2. **Part of that rise is an artifact of the global "between" set.** Comparing siblings only lowers basic-level d (CLIP pre 1.80 to 1.36, behavior 1.58 to 0.93, WordNet 1.28 to 0.79, `sem_km` 2.85 to 1.36).
3. **H1b's ordering does not hold under the nested contrast for any data-derived RDM** (image-bootstrap 95% CIs of d1 - d2 and d2 - d3). CLIP and pixels peak at the basic level (CLIP pre d2 - d3 = -0.63 [-0.78, -0.47]); behavior (raw and both MDS embeddings) and WordNet peak at the mid-level (raw behavior d1 - d2 = -0.33 [-0.40, -0.26]; MDS low -0.90 [-1.11, -0.72], MDS high -0.83 [-1.00, -0.67]; WordNet -0.46 [-0.57, -0.35]). The embeddings do satisfy the second step (d2 - d3 = 0.53 [0.28, 0.79] low, 0.42 [0.18, 0.65] high), so behavior is higher at the mid-level than at the basic level, but not higher at the domain level. Only `sem_km` follows both steps (0.24 [0.06, 0.40]; 1.20 [1.02, 1.38]).
4. **Per category (descriptive):** the weakest basic-level category is a heterogeneous catch-all (`inanimate/handmade/other`) for CLIP, behavior (raw and both embeddings: -0.16 low, -0.04 high); pixels and WordNet fail in different categories.

**Limitations.** (a) d is not strictly comparable across levels: the number and size of the groups and the heterogeneity of the between group change with L, and the level-1 "within" group contains all finer structure, which deflates top-level d for any RDM that clusters at finer levels. An ordering of d by level is therefore a weak test of H1b as worded. (b) Pair counts differ by level (nested between pairs: 49,272 to 64,696 at levels 2 and 3; within pairs: 21,236 to 135,204). (c) Raw behavioral d uses pairwise-complete pairs only; the MDS embeddings are complete but their larger d partly reflects the smoothing of the embedding, and the mid-level peak is the same at d = 5 and d = 20.